In [13]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
import json
import re 
import math
import concurrent.futures

def parse_product(article):
    """Takes a single BeautifulSoup <article> tag and returns a dictionary of data."""
    product_dict = {}
    
    # Get the ID directly from the article tag
    product_dict['id'] = article.get('id')
    
    # Brand
    brand = article.find('span', class_='name')
    product_dict['brand'] = brand.text.strip() if brand else None
    
    # Product Name (using a lambda to catch dynamic IDs like product-name-100019928)
    name = article.find('span', attrs={'data-testid': lambda x: x and x.startswith('product-name')})
    product_dict['name'] = name.text.strip() if name else None
    
    # Version/Detail
    version = article.find('span', attrs={'data-cy': lambda x: x and x.startswith('product-versioning')})
    product_dict['version'] = version.text.strip() if version else None
    
    # Price
    price = article.find('span', {'data-testid': 'current-price'})
    if price:
        try:
            # Convert "1.95" to a float number
            product_dict['price'] = float(price.text.strip())
        except ValueError:
            product_dict['price'] = price.text.strip()
    else:
        product_dict['price'] = None
        
    # Weight
    weight = article.find('span', {'data-testid': 'default-product-size'})
    product_dict['weight'] = weight.text.strip() if weight else None
    
    # Price per Unit
    price_unit = article.find('span', id=lambda x: x and x.endswith('-price-unit'))
    product_dict['price_per_unit'] = price_unit.text.strip() if price_unit else None
    
    # Image URL
    img = article.find('img')
    product_dict['image_url'] = img.get('src') if img else None
    
    return product_dict



In [14]:
def create_driver():
    print("Setting up the browser...")
    chrome_options = Options()
    chrome_options.add_argument("--headless=new") 
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver


def get_remaining_products_count(driver,url):
    rendered_html = get_rendered_page_sync(driver, url)
    soup = BeautifulSoup(rendered_html, 'html.parser')
    remaining_div = soup.find('div', class_='remaining-products')
    if remaining_div:
        raw_text = remaining_div.text.strip() 
        clean_numbers = re.sub(r'\D', '', raw_text) 
        if clean_numbers:
            return math.ceil(int(clean_numbers) / 100)

    return 0

# --- 2. The Selenium Scraper ---
def get_rendered_page_sync(driver,url): 
    try:
        print(f"Loading {url}...")
        driver.get(url)
        
        # Give Angular enough time to build the list of products
        print("Waiting for products to load...")
        time.sleep(2) 
        
        html_content = driver.page_source
        return html_content
    except Exception as e:
        print(f"Error loading page: {e}")
        return None


def scrape_single_page(url):
    """This function runs in parallel. It MUST create its own driver."""
    print(f"Thread starting for: {url}")
    
    # Create a unique driver for this specific thread
    driver = create_driver()
    page_products = []
    
    try:
        # Step A: Get the HTML
        rendered_html = get_rendered_page_sync(driver, url)
        
        if rendered_html:
            soup = BeautifulSoup(rendered_html, 'html.parser')
            
            # Step C: Find ALL product cards
            product_cards = soup.find_all('article', class_='product-card')
            
            # Step D: Loop through them and parse
            for card in product_cards:
                parsed_data = parse_product(card)
                if parsed_data and parsed_data.get('name'):
                    page_products.append(parsed_data)
                    
        return page_products # Return the list of dictionaries for this page

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return []
        
    finally:
        # CRITICAL: Always close the thread's driver so you don't run out of RAM!
        driver.quit()


In [ ]:
# --- 2. Main Execution ---
if __name__ == "__main__":
    
    temp_driver = create_driver()
    base_url = "https://www.migros.ch/fr/category/pates-condiments-conserves"
    
    # Calculate pages
    number_of_pages = get_remaining_products_count(temp_driver, base_url) + 1
    temp_driver.quit() # Close the temp driver
    
    # Generate all URLs
    urls = [f"{base_url}?page={i}" for i in range(1, int(number_of_pages) + 1)]
    print(f"Total pages to scrape: {len(urls)}")

    all_products = []
    
    # --- PARALLEL EXECUTION ---

    MAX_WORKERS = 50
    
    print(f"Starting parallel scraping with {MAX_WORKERS} workers...")
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = executor.map(scrape_single_page, urls)
        
        for page_results in results:
            if page_results:
                # Add the products from this page to our master list
                all_products.extend(page_results)

    print(f"\nSuccessfully extracted {len(all_products)} total products.")
    
    print("\n--- Sample of Extracted Data ---")
    print(json.dumps(all_products[:3], indent=4, ensure_ascii=False))
    
    with open("migros_products.json", "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=4, ensure_ascii=False)
        
    print("\nAll data successfully saved to 'migros_products.json'!")

Setting up the browser...
Loading https://www.migros.ch/fr/category/pates-condiments-conserves...
Waiting for products to load...
Total pages to scrape: 17
Starting parallel scraping with 20 workers...
Thread starting for: https://www.migros.ch/fr/category/pates-condiments-conserves?page=1
Setting up the browser...
Thread starting for: https://www.migros.ch/fr/category/pates-condiments-conserves?page=2
Setting up the browser...
Thread starting for: https://www.migros.ch/fr/category/pates-condiments-conserves?page=3
Setting up the browser...
Thread starting for: https://www.migros.ch/fr/category/pates-condiments-conserves?page=4
Setting up the browser...
Thread starting for: https://www.migros.ch/fr/category/pates-condiments-conserves?page=5
Setting up the browser...
Thread starting for: https://www.migros.ch/fr/category/pates-condiments-conserves?page=6
Setting up the browser...
Thread starting for: https://www.migros.ch/fr/category/pates-condiments-conserves?page=7
Setting up the brow